# Playground Series S6E4 - Reuse CatBoost Tuning Submission

This notebook is a lightweight submission workflow. It reuses the output from the long CatBoost tuning run instead of retraining models.

Source tuning notebook:
https://www.kaggle.com/code/tuannm3823/s6e4-predicting-irrigation-need-catboost-tuning?scriptVersionId=320060317

## 1. Setup

Attach the output of the CatBoost tuning notebook as an input dataset in Kaggle. This notebook searches `/kaggle/input` for a generated `submission.csv`, validates it against the competition sample submission, and writes a fresh `/kaggle/working/submission.csv`.

In [ ]:
from pathlib import Path
from IPython.display import display

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.5f}".format)

INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
TUNING_NOTEBOOK_SLUG = "s6e4-predicting-irrigation-need-catboost-tuning"

if not INPUT_ROOT.exists():
    raise RuntimeError(
        "This notebook is intended to run on Kaggle, where /kaggle/input is available."
    )

## 2. Locate Competition and Tuning Outputs

The competition sample submission defines the required IDs and columns. The tuning output should contain the generated `submission.csv` from the selected CatBoost experiment.

In [ ]:
def find_file(
    filename: str, prefer_slug: str | None = None, exclude_competition: bool = False
) -> Path:
    """Return a Kaggle input file, optionally preferring a notebook output.

    Args:
        filename: File name to search for below `/kaggle/input`.
        prefer_slug: Optional Kaggle dataset or notebook slug to prefer.
        exclude_competition: Whether to ignore competition-mounted files.

    Returns:
        Path to the selected input file.
    """
    matches = sorted(INPUT_ROOT.rglob(filename))
    if exclude_competition:
        matches = [p for p in matches if "competitions" not in p.parts]
    if prefer_slug:
        preferred = [p for p in matches if prefer_slug in str(p)]
        if preferred:
            return preferred[0]
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {INPUT_ROOT}.")
    return matches[0]


sample_submission_path = find_file("sample_submission.csv")
try:
    tuning_submission_path = find_file(
        "submission.csv", prefer_slug=TUNING_NOTEBOOK_SLUG, exclude_competition=True
    )
except FileNotFoundError:
    all_csvs = sorted(INPUT_ROOT.rglob("*.csv"))
    print("Available CSV files:")
    for path in all_csvs:
        print(" -", path)
    raise

print("sample submission:", sample_submission_path)
print("tuning submission:", tuning_submission_path)

## 3. Validate Submission

Validate row count, column names, IDs, missing predictions, and prediction labels before writing the final submission file.

In [ ]:
sample_submission = pd.read_csv(sample_submission_path)
tuning_submission = pd.read_csv(tuning_submission_path)

id_col = sample_submission.columns[0]
target_col = sample_submission.columns[-1]

print("sample shape:", sample_submission.shape)
print("tuning submission shape:", tuning_submission.shape)
print("required columns:", sample_submission.columns.tolist())
print("submission columns:", tuning_submission.columns.tolist())

if tuning_submission.shape[0] != sample_submission.shape[0]:
    raise ValueError(
        f"Row count mismatch: expected {sample_submission.shape[0]}, "
        f"got {tuning_submission.shape[0]}"
    )

if list(tuning_submission.columns) != list(sample_submission.columns):
    raise ValueError(
        "Column mismatch. The tuning submission must match sample_submission columns exactly."
    )

if not tuning_submission[id_col].equals(sample_submission[id_col]):
    raise ValueError(
        "ID order mismatch between tuning submission and sample submission."
    )

if tuning_submission[target_col].isna().any():
    raise ValueError("Submission contains missing predictions.")

allowed_labels = {"Low", "Medium", "High"}
observed_labels = set(tuning_submission[target_col].dropna().unique())
extra_labels = observed_labels - allowed_labels
if extra_labels:
    raise ValueError(f"Unexpected prediction labels: {sorted(extra_labels)}")

print("Validation passed.")
display(tuning_submission.head())
display(
    tuning_submission[target_col]
    .value_counts(normalize=True)
    .mul(100)
    .rename("prediction_pct")
    .to_frame()
)

## 4. Write Final Submission

Copy the validated tuning submission into `/kaggle/working/submission.csv`. This is the file Kaggle will expose for download and submission.

In [ ]:
output_path = WORKING_DIR / "submission.csv"
tuning_submission.to_csv(output_path, index=False)
print(f"Wrote {output_path}")

## 5. Submission Summary

The reused tuning output was validated successfully and written to `/kaggle/working/submission.csv`.

Current submission result:

- Public leaderboard score: `0.96094`.
- Source: output from `s6e4-predicting-irrigation-need-catboost-tuning`.
- Validation: `270,000` rows, expected columns, matching ID order, no missing predictions, and labels limited to `Low`, `Medium`, and `High`.
- Prediction mix: `Low 59.23%`, `Medium 37.59%`, `High 3.18%`.

Use this notebook for fast resubmission or validation of the saved tuning output. Only rerun the full tuning notebook when testing a new modeling idea.